<a href="https://colab.research.google.com/github/jayw20230711/AI-DataSets/blob/main/training_deep_network_section_3_tpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

## STILL MEMORY CRASH - DON'T RUN THIS CELL
import sys
import os
import gc
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch_xla
import torch_xla.core.xla_model as xm

# 1. Environment Tweaks for Stability
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
gc.collect()

# 2. Device Setup & Hard Check
device = torch_xla.device()
if device.type != 'xla':
    print(f"CRITICAL ERROR: TPU NOT FOUND! Detected: {device.type}")
    sys.exit("Stopping execution.")

print(f"TPU Verified: {device}")

# ── Step 1: Dataloaders (Memory Optimized) ────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.RandomResizedCrop(128),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = T.Compose([
    T.Resize(128),
    T.CenterCrop(128),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
val_dataset   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transforms)

# STABILITY FIX: num_workers=0 and pin_memory=False prevents Host RAM crashes in Colab
# drop_last=True prevents a second XLA compilation at the end of the epoch
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False,
                          num_workers=0, pin_memory=False, drop_last=True)

# ── Step 2: Build the model ───────────────────────────────────────
model = torchvision.models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# ── Step 3: Train function ────────────────────────────────────────
def train(model, train_loader, val_loader, num_epochs=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

    total_steps  = num_epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    for epoch in range(num_epochs):
        model.train()
        for i, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            with torch.autocast(device_type='xla', dtype=torch.bfloat16):
                logits = model(x)
                loss   = criterion(logits, y)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # The critical TPU sync point
            xm.optimizer_step(optimizer)
            scheduler.step()

            if i % 100 == 0:
                print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

        # Validation
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type='xla', dtype=torch.bfloat16):
                    preds = model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total   += len(y)

        print(f"Epoch {epoch+1} Complete | Val Acc: {correct/total:.3f}")

        # Save to Drive (Ensure Drive is mounted)
        xm.save(model.state_dict(), 'resnet50_cifar10.pth')

# ── Step 4: Execute ───────────────────────────────────────────────
train(model, train_loader, val_loader, num_epochs=10)

TPU Verified: xla:0


100%|██████████| 170M/170M [00:05<00:00, 29.0MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 343MB/s]


Epoch 1 | Batch 0/390 | Loss: 2.3093


In [ ]:
"""
DON;T RUN THIS CELL

If you are seeing Epoch 0 | Batch 0, it means the compilation finished and the TPU has actually started "ticking." However, 6 minutes without Batch 50 appearing means you are likely suffering from Data Starvation.

Because we set num_workers=0 to save RAM, the CPU is struggling to resize and normalize images one-by-one fast enough to feed the TPU. The TPU is sitting idle waiting for data.

How to break the bottleneck
To fix this without crashing the RAM, we need to find the "sweet spot" for data loading. Since you are on a TPU v5e, you have a bit more power than the old v2.

Make these 3 specific changes to your current script:

Increase num_workers to 2: This allows two separate CPU threads to prep images while the TPU is busy. It uses more RAM, but it's the only way to speed up the batches.

Increase batch_size back to 128: Small batches (32) actually make TPUs slower because the overhead of sending data to the hardware outweighs the computation time.

Disable autocast for now: Sometimes the overhead of Bfloat16 conversion on the CPU side slows down the first few epochs.


"""


import sys
import os
import gc
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch_xla
import torch_xla.core.xla_model as xm

# 1. Force Python to give back memory immediately
gc.collect()

# 2. Device Check
device = torch_xla.device()
if device.type != 'xla':
    sys.exit("TPU not found. Stopping.")

# ── Step 1: Dataloaders (Ultra-Light) ────────────────────────
# Reduced image size to 64 to lower the RAM footprint during training
train_transforms = T.Compose([
    T.Resize(64),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Loading dataset
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
val_ds   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=train_transforms)

# CRITICAL: Small batch size to save RAM during the "Batch 0" compilation
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=False, drop_last=True)

# Free dataset memory from CPU RAM
del train_ds
del val_ds
gc.collect()

# ── Step 2: Model (Late Loading) ──────────────────────────────
# Load the model structure
model = torchvision.models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, 10)

# Move to TPU and immediately clear CPU reference
model = model.to(device)
gc.collect()

# ── Step 3: Train ─────────────────────────────────────────────
def train():
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        model.train()
        for i, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            with torch.autocast(device_type='xla', dtype=torch.bfloat16):
                loss = criterion(model(x), y)

            loss.backward()
            xm.optimizer_step(optimizer)

            if i % 50 == 0:
                print(f"Epoch {epoch} | Batch {i} | Loss {loss.item():.4f}")

        # Save after every epoch to prevent loss if it crashes later
        xm.save(model.state_dict(), 'checkpoint.pth')

# Clear everything again before starting
gc.collect()
train()

100%|██████████| 170M/170M [00:03<00:00, 48.7MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 257MB/s]


Epoch 0 | Batch 0 | Loss 2.2797


In [ ]:
"""
DON'T RUN THIS CELL

This is the hardened, memory-optimized version of your code. I have balanced the RAM usage (to prevent the Colab crash) with the data loading speed (to prevent the "6-minute wait").

I have set num_workers=2 and batch_size=128. This is the "Goldilocks" zone for a TPU v5e in Colab.

"""
import sys
import os
import gc
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch_xla
import torch_xla.core.xla_model as xm

# 1. Environment & RAM Cleanup
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
gc.collect()

# 2. Device Setup (New recommended path)
device = torch_xla.device()
if device.type != 'xla':
    print("\n" + "!"*60)
    print("CRITICAL ERROR: TPU NOT DETECTED!")
    print("Go to: Edit > Notebook settings > Hardware accelerator > TPU v5e")
    print("!"*60 + "\n")
    sys.exit("Execution stopped.")

print(f"TPU Verified: {device}")

# 3. Data Preparation (RAM Optimized)
# We use 128x128 for a balance of speed and ResNet compatibility
stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
transform = T.Compose([
    T.Resize(64),
    T.ToTensor(),
    T.Normalize(*stats),
])

print("Downloading and preparing data...")
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
val_ds   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# num_workers=2 provides speed without duplicating too much RAM
# drop_last=True is MANDATORY for TPU stability
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False,
                          num_workers=4, pin_memory=False, drop_last=True)

# Free dataset memory from Host RAM
del train_ds, val_ds
gc.collect()

# 4. Model Setup
print("Building ResNet50...")
model = torchvision.models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# 5. Training Logic
def train_model(model, train_loader, val_loader, epochs=5):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

    for epoch in range(epochs):
        model.train()
        print(f"\nStarting Epoch {epoch+1}")

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Using bfloat16 for TPU efficiency
            with torch.autocast(device_type='xla', dtype=torch.bfloat16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()

            # CRITICAL: This is the specific TPU optimizer step
            xm.optimizer_step(optimizer)

            if i % 20 == 0:
                print(f"Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

        # Validation at end of epoch
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with torch.autocast(device_type='xla', dtype=torch.bfloat16):
                    outputs = model(images)
                    preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        print(f"Epoch {epoch+1} Complete | Accuracy: {100 * correct / total:.2f}%")

        # Save checkpoint using XLA-safe method
        xm.save(model.state_dict(), f'resnet50_cifar10_ep{epoch+1}.pth')

# 6. Execute
if __name__ == '__main__':
    gc.collect()
    train_model(model, train_loader, val_loader)


TPU Verified: xla:0


100%|██████████| 170M/170M [00:03<00:00, 49.0MB/s]


Building ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 286MB/s]



Starting Epoch 1
Batch 0/390 | Loss: 2.3169


In [ ]:
"""
Let's do a Minimalist Reset. This code removes the ImageNet weights
(the biggest RAM hog) and uses the simplest possible setup to prove
the TPU is alive.
"""

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch_xla
import torch_xla.core.xla_model as xm
import gc

# 1. IMMEDIATE CLEANUP
gc.collect()
device = torch_xla.device()

# 2. NO RESIZING, NO DOWNLOADS (32x32 native)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# We download only once. If already downloaded, it's instant.
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# num_workers=0 is the "Safe Mode" to ensure no RAM spikes
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)

# 3. TINY MODEL (ResNet18 instead of 50 to save RAM/Compilation time)
print("Building ResNet18 (Fastest compilation)...")
model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# 4. THE STRIPPED-DOWN LOOP
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("TPU is compiling... If this takes > 2 mins, restart your runtime.")
model.train()
for i, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()

    xm.optimizer_step(optimizer)

    # Force a print on the very first successful step
    print(f"STEP {i} COMPLETED | Loss: {loss.item():.4f}")
    if i > 100: break

100%|██████████| 170M/170M [00:04<00:00, 41.2MB/s]


Building ResNet18 (Fastest compilation)...
TPU is compiling... If this takes > 2 mins, restart your runtime.
STEP 0 COMPLETED | Loss: 2.4687
STEP 1 COMPLETED | Loss: 2.5250
STEP 2 COMPLETED | Loss: 2.4745
STEP 3 COMPLETED | Loss: 2.3472
STEP 4 COMPLETED | Loss: 2.3552
STEP 5 COMPLETED | Loss: 2.3179
STEP 6 COMPLETED | Loss: 2.2157
STEP 7 COMPLETED | Loss: 2.2308
STEP 8 COMPLETED | Loss: 2.2934
STEP 9 COMPLETED | Loss: 2.3292
STEP 10 COMPLETED | Loss: 2.2727
STEP 11 COMPLETED | Loss: 2.1277
STEP 12 COMPLETED | Loss: 2.2788
STEP 13 COMPLETED | Loss: 2.1288


In [ ]:
"""
To hit that 1-step-per-second mark, we have to bypass the slow CPU-to-TPU
transfer. The fastest way to do this on a TPU is to use Synthetic Data first.

If synthetic data runs fast, we know the TPU is fine and the problem is purely
your disk/CPU loading CIFAR images. If synthetic data is also slow, the XLA
software layer is hung and you need a new runtime.

Run this exact block. It generates random tensors directly on the TPU device,
removing the "data loading" bottleneck entirely.

"""

import torch
import torch.nn as nn
import torchvision
import torch_xla.core.xla_model as xm
import torch_xla
import time

device = torch_xla.device()
print(f"Hardware: {device}")

# 1. Light Model
model = torchvision.models.resnet18(weights=None).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 2. Synthetic Data (Zero Disk/CPU usage)
# Creating 128 images of 32x32 directly in memory
images = torch.randn(128, 3, 32, 32).to(device)
labels = torch.randint(0, 10, (128,)).to(device)

print("Starting Speed Test (Synthetic)...")
model.train()

start_time = time.time()

for i in range(50):
    optimizer.zero_grad()
    output = model(images)
    loss = criterion(output, labels)
    loss.backward()
    xm.optimizer_step(optimizer)

    if i % 10 == 0:
        elapsed = time.time() - start_time
        print(f"Step {i} | Time elapsed: {elapsed:.2f}s")

print(f"\nFinal Speed: {50 / (time.time() - start_time):.2f} steps/second")

Hardware: xla:0
Starting Speed Test (Synthetic)...
Step 0 | Time elapsed: 0.06s
Step 10 | Time elapsed: 0.63s
Step 20 | Time elapsed: 1.18s
Step 30 | Time elapsed: 1.75s
Step 40 | Time elapsed: 2.30s

Final Speed: 17.90 steps/second


In [ ]:
"""
The Solution: "In-Memory" Training
Since CIFAR-10 is tiny (only about 160MB total), the best way to hit
that 1-step-per-second mark with real data is to load the entire dataset
into RAM once, and then feed it to the TPU. This skips the slow disk-reading
process entirely.
"""

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torch_xla.core.xla_model as xm
import torch_xla
import gc

# 1. Hardware & Cleanup
device = torch_xla.device()
gc.collect()
print(f"Hardware: {device}")

# 2. Data Preparation (Load to RAM)
print("Loading CIFAR-10 into memory...")
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Download and load full dataset
full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# Convert entire dataset to a single tensor on the TPU to bypass CPU bottlenecks
# This fits easily in TPU memory (~150MB)
print("Transferring data to TPU...")
all_images = torch.stack([img for img, _ in full_train]).to(device)
all_labels = torch.tensor([lbl for _, lbl in full_train]).to(device)

# 3. Model & Optimizer
print("Building ResNet18...")
model = torchvision.models.resnet18(weights=None).to(device)
model.fc = nn.Linear(model.fc.in_features, 10).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 4. Training Loop
print("\nStarting Training (In-Memory)...")
model.train()
batch_size = 128

for step in range(501):
    # Randomly sample a batch from the tensors in memory
    indices = torch.randint(0, len(all_images), (batch_size,))
    img_batch = all_images[indices]
    lbl_batch = all_labels[indices]

    optimizer.zero_grad()
    outputs = model(img_batch)
    loss = criterion(outputs, lbl_batch)
    loss.backward()

    # Crucial XLA optimizer step
    xm.optimizer_step(optimizer)

    if step % 50 == 0:
        # loss.item() is okay here because we are only doing it every 50 steps
        print(f"Step {step:3} | Loss: {loss.item():.4f}")

print("\nTraining complete. Hardware utilized successfully.")




Hardware: xla:0
Loading CIFAR-10 into memory...


100%|██████████| 170M/170M [00:05<00:00, 28.9MB/s]


Transferring data to TPU...
Building ResNet18...

Starting Training (In-Memory)...
Step   0 | Loss: 2.5814
Step  50 | Loss: 1.6016


In [ ]:
"""
If it has been 6 minutes and you are still stuck at Step 50, your training
has likely "stalled" due to a graph recompilation error.

The Problem: Random Sampling in the Loop
The line indices = torch.randint(...) inside the loop might be causing XLA
to think the graph is dynamic.

The Final, Bulletproof Fix (Static Graph)
To fix this, we move the "batching" logic outside of the XLA graph by using
a standard DataLoader with num_workers=0 but feeding it the In-Memory tensors.
This keeps the graph perfectly static and the data flow local.
"""


import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torch_xla.core.xla_model as xm
import torch_xla
from torch.utils.data import DataLoader, TensorDataset
import gc

# 1. Setup
device = torch_xla.device()
print(f"Hardware: {device}")

# 2. Load and Prepare Tensors (Defining all_images and all_labels here)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

print("Downloading and loading data into RAM...")
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# Combine dataset into single tensors to bypass disk bottlenecks
all_images = torch.stack([img for img, _ in train_ds]).to(device)
all_labels = torch.tensor([lbl for _, lbl in train_ds]).to(device)

# Create a fast loader using the tensors in memory
train_set = TensorDataset(all_images, all_labels)
fast_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)

# 3. Model Setup
model = torchvision.models.resnet18(weights=None).to(device)
model.fc = nn.Linear(model.fc.in_features, 10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 4. Training Loop
print("Starting Training...")
model.train()

for i, (img, lbl) in enumerate(fast_loader):
    img, lbl = img.to(device), lbl.to(device)

    optimizer.zero_grad()
    outputs = model(img)
    loss = criterion(outputs, lbl)
    loss.backward()
    xm.optimizer_step(optimizer)

    if i % 10 == 0:
        print(f"Step {i} | Loss: {loss.item():.4f}")

    if i >= 100: # Short test run to verify speed
        break

Hardware: xla:0


100%|██████████| 170M/170M [00:03<00:00, 44.5MB/s]


Starting Training...
Step 0 | Loss: 2.5581
Step 10 | Loss: 2.0393
Step 20 | Loss: 2.0473


In [ ]:
"""
Since the Synthetic Data test hit 17 steps per second, we know the hardware is
capable. If the "In-Memory" version is this slow, it means the XLA
compilation is triggering on every single step because the graph isn't
staying static.

The Problem: The "item()" Trap
In the code, loss.item() is a "blocking" call. It forces the TPU to stop,
calculate the loss, send it back to the CPU, and wait. On some Colab instances,
this handshake is broken and causes a massive delay.

"""
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torch_xla.core.xla_model as xm
import torch_xla
from torch.utils.data import DataLoader, TensorDataset
import time

# 1. HARDWARE INIT
device = torch_xla.device()
print(f"Using Hardware: {device}")

# 2. DATA LOADING (IN-MEMORY STRATEGY)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

print("Downloading CIFAR-10...")
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

print("Moving dataset to TPU RAM (Bypassing Disk)...")
# Loading everything into a single tensor on the device
all_images = torch.stack([img for img, _ in train_ds]).to(device)
all_labels = torch.tensor([lbl for _, lbl in train_ds]).to(device)

train_set = TensorDataset(all_images, all_labels)
# num_workers=0 is MANDATORY here because data is already on device
fast_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)

# 3. MODEL SETUP
print("Building Model...")
model = torchvision.models.resnet18(weights=None).to(device)
model.fc = nn.Linear(model.fc.in_features, 10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 4. HIGH-SPEED TRAINING LOOP
print("\nStarting Training. First 10 steps will compile (slow), then it will fly.")
model.train()
start_time = time.time()

for i, (img, lbl) in enumerate(fast_loader):
    # These are already on device, but we keep this for XLA graph stability
    img, lbl = img.to(device), lbl.to(device)

    optimizer.zero_grad()
    outputs = model(img)
    loss = criterion(outputs, lbl)
    loss.backward()

    # Execution barrier - this sends the graph to the TPU
    xm.optimizer_step(optimizer)

    # DO NOT call loss.item() every step. It kills performance.
    if i % 20 == 0 and i > 0:
        elapsed = time.time() - start_time
        print(f"Step {i} | Time since start: {elapsed:.2f}s")

    if i >= 200: # Limit to 200 steps for this speed test
        break

print(f"\nFinal Time for 200 steps: {time.time() - start_time:.2f}s")


Using Hardware: xla:0


100%|██████████| 170M/170M [00:06<00:00, 25.9MB/s]


Moving dataset to TPU RAM (Bypassing Disk)...
Building Model...

Starting Training. First 10 steps will compile (slow), then it will fly.
Step 20 | Time since start: 1.25s
Step 40 | Time since start: 2.43s
Step 60 | Time since start: 3.63s
Step 80 | Time since start: 4.84s
Step 100 | Time since start: 6.05s
Step 120 | Time since start: 7.27s
Step 140 | Time since start: 8.48s
Step 160 | Time since start: 9.67s
Step 180 | Time since start: 10.86s
Step 200 | Time since start: 12.04s

Final Time for 200 steps: 12.04s


In [ ]:
import sys
import os
import gc
import time
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch_xla
import torch_xla.core.xla_model as xm

# 1. Clear everything and set environment
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
gc.collect()

device = torch_xla.device()
print(f"TPU Verified: {device}")

# 2. Ultra-Light Data Loading (32x32 = No CPU Resizing overhead)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
val_ds   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# 4 workers at 32x32 should be lightning fast
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False,
                          num_workers=4, pin_memory=False, drop_last=True)

del train_ds, val_ds
gc.collect()

# 3. Model Setup (Load weights on CPU first to save TPU memory)
print("Loading Model...")
model = torchvision.models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device) # Move to TPU only after setup is done

# 4. Stabilized Training Loop
def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    print("Pre-warming system... Starting Batch 0 compilation.")
    model.train()

    for epoch in range(5):
        for i, (images, labels) in enumerate(train_loader):
            # Move data to TPU
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Use standard Float32 for Batch 0 to ensure successful compilation
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            xm.optimizer_step(optimizer)

            # Print every 5 batches initially so you know it's alive
            if i % 5 == 0:
                print(f"Epoch {epoch} | Batch {i} | Loss: {loss.item():.4f}")

if __name__ == '__main__':
    time.sleep(2) # Give Colab a second to breathe
    gc.collect()
    train_model(model, train_loader, val_loader)

TPU Verified: xla:0


100%|██████████| 170M/170M [00:03<00:00, 47.2MB/s]


Loading Model...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 237MB/s]


Pre-warming system... Starting Batch 0 compilation.
Epoch 0 | Batch 0 | Loss: 2.3584
Epoch 0 | Batch 5 | Loss: 2.3326
Epoch 0 | Batch 10 | Loss: 2.2945
Epoch 0 | Batch 15 | Loss: 2.2783
Epoch 0 | Batch 20 | Loss: 2.2603


In [ ]:
import os
print("Current Directory:", os.getcwd())
print("Files in this folder:", os.listdir())

In [ ]:
# Force the Download
from google.colab import files
import os

filename = 'resnet50_cifar10.pth'
if os.path.exists(filename):
    files.download(filename)
else:
    # Try the most common absolute path in Colab
    alt_path = os.path.join('/content', filename)
    if os.path.exists(alt_path):
        files.download(alt_path)
    else:
        print("Still can't find the file. Are you sure the training loop reached the 'save' line?")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive synced and unmounted. Your 96.8% model is safe!")